# FABLE-5 — Budget-Aware Melanoma Lesion Retrieval (Colab Runbook)

Runs the whole engine in order: setup → data → audit → build → embeddings → features → train/eval seed 42 → multiseed → error analysis → zip.

The core research engine (features, all rankers, evaluation, bootstrap) is pure numpy/sklearn and runs on CPU. A GPU only speeds up **embedding extraction** (Step 5). Leave the real-data paths empty to run on synthetic ISIC-shaped data.

## Rationale & literature

FABLE-5 reframes melanoma screening as **patient-level lesion retrieval under a review budget**: rank each patient's lesions so malignant ones surface early, rather than scoring lesions independently. The design draws on:

- **Rotemberg et al.**, *A Patient-Centric Dataset of Images and Metadata for Identifying Melanomas Using Clinical Context* — motivates using per-patient context rather than isolated lesions.
- **Al Zegair et al.**, *Application of Machine Learning in Melanoma Detection and the Identification of Ugly Duckling and Suspicious Naevi* — the within-patient 'ugly duckling' outlier idea behind the group-B context features.
- **Pedregosa et al.**, *Learning to Rank from Medical Imaging Data* — precedent for optimizing a ranking objective on medical images.
- **Burges et al.**, *Learning to Rank Using Gradient Descent* (RankNet) — basis for the pairwise logistic / MLP rankers.
- **Burges**, *From RankNet to LambdaRank to LambdaMART: An Overview* — basis for the lambda-weighted pairwise ranker.
- **iToBoS** total-body dataset and **ISIC 2024 / SLICE-3D** — the total-body photography setting and data source.

AUROC/AUPRC are reported but secondary; the primary metrics are recall@k, first-malignant rank, number-needed-to-review, and normalized percentile.

## Step 1 — Setup

In [ ]:
# Install light deps (torch/torchvision are preinstalled on Colab GPU runtimes).
!pip -q install pyyaml pyarrow scikit-learn scipy matplotlib h5py >/dev/null
# Optional DINO/timm extractor (only needed if USE_DINO=True below):
# !pip -q install timm >/dev/null
import os, sys, subprocess, json
print('python', sys.version.split()[0])

In [ ]:
# Get the FABLE-5 code into /content. Option A: clone your repo. Option B: upload.
REPO_DIR = '/content/fable5_budget_aware_retrieval'
if not os.path.isdir(REPO_DIR):
    # Option A (edit to your repo):
    # !git clone https://github.com/<you>/highres-candidate-reranking.git /content/repo
    # REPO_DIR = '/content/repo/fable5_budget_aware_retrieval'
    #
    # Option B: upload the folder as a zip, then unzip:
    from google.colab import files          # noqa
    up = files.upload()                       # choose fable5_budget_aware_retrieval.zip
    zname = next(iter(up))
    !unzip -o -q "$zname" -d /content
assert os.path.isdir(REPO_DIR), REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())

## Runtime-safety flags
Everything below is driven by these flags. They are compiled into a generated config (`configs/_colab.yaml`) so the `run_*` scripts stay unchanged.

In [ ]:
RUN_MODE            = 'SMOKE'     # 'SMOKE' | 'MEDIUM' | 'FULL'
USE_DINO            = False       # optional timm/DINO extractor (needs `pip install timm`)
RUN_PAIRWISE        = True        # include pairwise rankers (logreg/mlp/lambda)
RUN_LISTWISE        = True        # include listwise softmax ranker
RUN_MULTISEED       = True        # all seeds vs single seed 42
MAX_LESIONS         = None        # override subset cap (None -> use RUN_MODE preset)
MAX_PATIENTS        = None        # override patient cap
EMBEDDING_BATCH_SIZE= 128         # lower if the GPU OOMs during extraction
EMBEDDING_MODEL     = 'resnet50'  # 'resnet50' | 'efficientnet_b0'

# Real ISIC 2024 data paths (leave as None to run on synthetic data):
METADATA_CSV = None   # e.g. '/content/isic2024/train-metadata.csv'
HDF5_PATH    = None   # e.g. '/content/isic2024/train-image.hdf5'

In [ ]:
# Memory check + FULL->MEDIUM fallback.
import psutil
avail_gb = psutil.virtual_memory().available / 1e9
print(f'available RAM: {avail_gb:.1f} GB')
if RUN_MODE == 'FULL' and avail_gb < 20:
    print('!! low RAM for FULL -> falling back to MEDIUM')
    RUN_MODE = 'MEDIUM'

In [ ]:
# Compile the flags into a generated config.
from src.config import load_config
import yaml
cfg = load_config(f'configs/fable5_{RUN_MODE.lower()}.yaml', {'run_mode': RUN_MODE})
cfg['embedding']['model'] = EMBEDDING_MODEL
cfg['embedding']['use_dino'] = USE_DINO
cfg['embedding']['batch_size'] = EMBEDDING_BATCH_SIZE
if MAX_LESIONS is not None: cfg['subset']['max_lesions'] = MAX_LESIONS
if MAX_PATIENTS is not None: cfg['subset']['max_patients'] = MAX_PATIENTS
if METADATA_CSV: cfg['paths']['metadata_csv'] = METADATA_CSV
if HDF5_PATH:    cfg['paths']['hdf5'] = HDF5_PATH
drop = set()
if not RUN_PAIRWISE: drop |= {'pairwise_rank_logreg','pairwise_rank_mlp','lambda_pairwise_logreg'}
if not RUN_LISTWISE: drop |= {'listwise_softmax_ranker'}
cfg['models'] = [m for m in cfg['models'] if m not in drop]
if not RUN_MULTISEED: cfg['seeds'] = [42]
with open('configs/_colab.yaml','w') as f: yaml.safe_dump(cfg, f)
CFG = 'configs/_colab.yaml'
print('models:', cfg['models']); print('seeds:', cfg['seeds']); print('run_mode:', cfg['run_mode'])

## Step 2 — Kaggle data download (skip for synthetic)
Upload your `kaggle.json` token, then pull the ISIC 2024 metadata + image HDF5. Set `METADATA_CSV` / `HDF5_PATH` above to the downloaded paths.

In [ ]:
if METADATA_CSV or HDF5_PATH:
    from google.colab import files
    print('upload kaggle.json'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle >/dev/null
    !mkdir -p /content/isic2024
    !kaggle competitions download -c isic-2024-challenge -p /content/isic2024
    !cd /content/isic2024 && unzip -o -q isic-2024-challenge.zip
    !ls -lh /content/isic2024 | head
else:
    print('No real paths set -> running on SYNTHETIC data.')

## Step 3 & 4 — Data audit + build splits (`run_01`)

In [ ]:
!python run_01_build_data.py --config $CFG

## Step 5 — Extract / cache embeddings (`run_02`)

In [ ]:
!python run_02_extract_embeddings.py --config $CFG

## Step 6 — Build features (`run_03`)

In [ ]:
!python run_03_build_features.py --config $CFG

## Step 7 — Train / evaluate seed 42 (`run_04`)

In [ ]:
!python run_04_train_eval.py --config $CFG --seed 42

## Step 8 — Multiseed run + readable report (`run_05`)

In [ ]:
if RUN_MULTISEED:
    !python run_05_multiseed.py --config $CFG
else:
    print('RUN_MULTISEED=False -> seed 42 only (run_04 already done).')

## Step 9 — Error analysis + plots (`run_06`)

In [ ]:
!python run_06_error_report.py --config $CFG --seed 42

In [ ]:
# Show the readable report + key plots inline.
from IPython.display import Markdown, Image, display
rep = 'results/readable_report.md'
if os.path.exists(rep): display(Markdown(open(rep).read()))
for p in ['results/plots/recall_at_k_curve.png',
          'results/plots/delta_rank_hist.png',
          'results/plots/method_comparison_bar.png']:
    if os.path.exists(p): display(Image(p))

## Step 10 — Zip outputs

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out = f'/content/fable5_results_{RUN_MODE}_{stamp}'
shutil.make_archive(out, 'zip', 'results')
print('wrote', out + '.zip')
try:
    from google.colab import files; files.download(out + '.zip')
except Exception as e:
    print('download skipped:', e)